# SSD-MobileNetV2 TFOD untuk bibit lele
Notebook ini memakai TensorFlow Object Detection API, dengan dataset YOLO lama yang di-copy ke folder baru supaya sumber asli tetap aman.


## Alur
1. Copy `dataset_split` ke `dataset_split_tfod`.
2. Buat `label_map.pbtxt`.
3. Konversi label YOLO ke TFRecord.
4. Compile proto TFOD.
5. Siapkan config SSD-MobileNetV2.
6. Training.
7. Export SavedModel dan TFLite.


In [ ]:
%pip install -q tensorflow==2.15.0 pillow lxml matplotlib pandas tf_slim pycocotools protobuf==4.25.3


In [12]:
from pathlib import Path
import shutil
import os
import re
import subprocess
from PIL import Image
import tensorflow as tf

ROOT = Path.cwd()
SRC_DATASET = ROOT / 'dataset_split'
TFOD_DATASET = ROOT / 'dataset_split_tfod'
WORK_DIR = ROOT / 'tfod_work'
ANNOT_DIR = WORK_DIR / 'annotations'
MODEL_DIR = WORK_DIR / 'models'
EXPORT_DIR = WORK_DIR / 'export'
for p in [WORK_DIR, ANNOT_DIR, MODEL_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if not SRC_DATASET.exists():
    raise FileNotFoundError(f'Sumber dataset tidak ditemukan: {SRC_DATASET}')

if not TFOD_DATASET.exists():
    shutil.copytree(SRC_DATASET, TFOD_DATASET)
    print(f'Copy dataset selesai -> {TFOD_DATASET}')
else:
    print(f'Dataset copy sudah ada -> {TFOD_DATASET}')


Dataset copy sudah ada -> d:\Kuliah\Skripsi\model-dataset-2-kelas\dataset_split_tfod


In [13]:
def count_pairs(split_name):
    img_dir = TFOD_DATASET / split_name / 'images'
    lbl_dir = TFOD_DATASET / split_name / 'labels'
    imgs = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
    lbls = sorted(lbl_dir.glob('*.txt'))
    return len(imgs), len(lbls)

for split in ['train', 'valid', 'test']:
    n_img, n_lbl = count_pairs(split)
    print(split, 'images=', n_img, 'labels=', n_lbl)

label_map_path = ANNOT_DIR / 'label_map.pbtxt'
label_map_path.write_text(
    "item {\n id: 1\n name: 'normal'\n}\n\nitem {\n id: 2\n name: 'anomali'\n}\n",
    encoding='utf-8'
)
print(f'Label map dibuat: {label_map_path}')


train images= 993 labels= 993
valid images= 286 labels= 286
test images= 144 labels= 144
Label map dibuat: d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\annotations\label_map.pbtxt


In [14]:
CLASS_NAME_TO_ID = {'normal': 1, 'anomali': 2}
YOLO_ID_TO_CLASS = {0: 1, 1: 2}

def yolo_line_to_bbox(line, img_w, img_h):
    cls_id, x_center, y_center, w, h = line.strip().split()[:5]
    cls_id = int(cls_id)
    if cls_id not in YOLO_ID_TO_CLASS:
        return None
    class_id = YOLO_ID_TO_CLASS[cls_id]
    x_center = float(x_center) * img_w
    y_center = float(y_center) * img_h
    w = float(w) * img_w
    h = float(h) * img_h
    xmin = max(0.0, (x_center - w / 2.0) / img_w)
    ymin = max(0.0, (y_center - h / 2.0) / img_h)
    xmax = min(1.0, (x_center + w / 2.0) / img_w)
    ymax = min(1.0, (y_center + h / 2.0) / img_h)
    return class_id, xmin, ymin, xmax, ymax

def create_tf_example(image_path, label_path):
    with tf.io.gfile.GFile(str(image_path), 'rb') as f:
        encoded_jpg = f.read()
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    width, height = image.size

    xmins, xmaxs, ymins, ymaxs = [], [], [], []
    classes_text, classes = [], []

    if label_path.exists():
        lines = label_path.read_text(encoding='utf-8').strip().splitlines()
        for line in lines:
            if not line.strip():
                continue
            parsed = yolo_line_to_bbox(line, width, height)
            if parsed is None:
                continue
            class_id, xmin, ymin, xmax, ymax = parsed
            xmins.append(xmin)
            xmaxs.append(xmax)
            ymins.append(ymin)
            ymaxs.append(ymax)
            classes.append(class_id)
            classes_text.append([k for k, v in CLASS_NAME_TO_ID.items() if v == class_id][0].encode('utf-8'))

    filename = image_path.name.encode('utf-8')
    image_format = image_path.suffix.replace('.', '').encode('utf-8')

    feature = {
        'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
        'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
        'image/filename': tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename])),
        'image/source_id': tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename])),
        'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_jpg])),
        'image/format': tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_format])),
        'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
        'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
        'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
        'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
        'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
        'image/object/class/label': tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))

def write_tfrecord(split_name, out_name):
    img_dir = TFOD_DATASET / split_name / 'images'
    lbl_dir = TFOD_DATASET / split_name / 'labels'
    out_path = ANNOT_DIR / out_name
    image_paths = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
    with tf.io.TFRecordWriter(str(out_path)) as writer:
        for image_path in image_paths:
            label_path = lbl_dir / f'{image_path.stem}.txt'
            writer.write(create_tf_example(image_path, label_path).SerializeToString())
    print(f'Selesai -> {out_path} | total image: {len(image_paths)}')

write_tfrecord('train', 'train.record')
write_tfrecord('valid', 'val.record')
write_tfrecord('test', 'test.record')


Selesai -> d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\annotations\train.record | total image: 993
Selesai -> d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\annotations\val.record | total image: 286
Selesai -> d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\annotations\test.record | total image: 144


In [15]:
repo = Path(r'D:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models')
research = repo / 'research'
protoc = Path(r'C:\Users\TROYY\anaconda3\envs\skripkir\Library\bin\protoc.exe')
if not protoc.exists():
    raise FileNotFoundError(protoc)
subprocess.run([str(protoc), '--version'], check=True)
subprocess.run(f'"{protoc}" object_detection/protos/*.proto --python_out=.', cwd=research, shell=True, check=True)
print('Proto compile selesai')


Proto compile selesai


In [16]:
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
EXTRACT_DIR = MODEL_DIR / MODEL_NAME
if not EXTRACT_DIR.exists():
    raise FileNotFoundError(f'Folder model tidak ditemukan: {EXTRACT_DIR}. Taruh hasil ekstrak model di folder itu dulu.')
base_config = EXTRACT_DIR / 'pipeline.config'
custom_config = ANNOT_DIR / 'ssd_mobilenet_v2_fpnlite_lele.config'
print(base_config)


d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\models\ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8\pipeline.config


In [17]:
config_text = base_config.read_text(encoding='utf-8')
config_text = re.sub(r'num_classes:\s*\d+', 'num_classes: 2', config_text, count=1)
config_text = re.sub(r'batch_size:\s*\d+', 'batch_size: 8', config_text, count=1)
config_text = re.sub(r'fine_tune_checkpoint:\s*"[^"]+"', f'fine_tune_checkpoint: "{(EXTRACT_DIR / "checkpoint" / "ckpt-0").as_posix()}"', config_text, count=1)
config_text = re.sub(r'fine_tune_checkpoint_type:\s*"[^"]+"', 'fine_tune_checkpoint_type: "detection"', config_text)
config_text = re.sub(r'label_map_path:\s*"[^"]+"', f'label_map_path: "{label_map_path.as_posix()}"', config_text)
config_text = re.sub(r'input_path:\s*"PATH_TO_BE_CONFIGURED"', f'input_path: "{(ANNOT_DIR / "train.record").as_posix()}"', config_text, count=1)
config_text = re.sub(r'input_path:\s*"PATH_TO_BE_CONFIGURED"', f'input_path: "{(ANNOT_DIR / "val.record").as_posix()}"', config_text, count=1)
custom_config.write_text(config_text, encoding='utf-8')
print(f'Custom config disimpan: {custom_config}')


Custom config disimpan: d:\Kuliah\Skripsi\model-dataset-2-kelas\tfod_work\annotations\ssd_mobilenet_v2_fpnlite_lele.config


In [18]:
import os
import subprocess
import sys
from pathlib import Path

config_path = ROOT / 'tfod_work' / 'annotations' / 'ssd_mobilenet_v2_fpnlite_lele.config'
model_dir = ROOT / 'tfod_work' / 'training'
model_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([r'D:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models\research', r'D:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models\research\slim'])
cmd = [sys.executable, str(ROOT / 'tf_models' / 'research' / 'object_detection' / 'model_main_tf2.py'), f'--pipeline_config_path={config_path.as_posix()}', f'--model_dir={model_dir.as_posix()}', '--alsologtostderr']
result = subprocess.run(cmd, env=env, cwd=str(ROOT), capture_output=True, text=True)
print('RETURN CODE:', result.returncode)
print('STDOUT:\n' + result.stdout)
print('STDERR:\n' + result.stderr)


RETURN CODE: 1
STDOUT:

STDERR:
2026-05-10 19:01:42.562891: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[05/10 19:01:46] tensorflow WARNING: From d:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models\research\object_detection\model_main_tf2.py:114: The name tf.app.run is deprecated. Please use tf.compat.v1.app.run instead.

2026-05-10 19:01:46.258191: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE SSE2 SSE3 SSE4.1 SSE4.2 AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
I05

In [19]:
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([r'D:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models\research', r'D:\Kuliah\Skripsi\model-dataset-2-kelas\tf_models\research\slim'])
export_dir = ROOT / 'tfod_work' / 'export' / 'ssd_savedmodel'
export_dir.mkdir(parents=True, exist_ok=True)
pipeline_config = ROOT / 'tfod_work' / 'annotations' / 'ssd_mobilenet_v2_fpnlite_lele.config'
trained_checkpoint_dir = ROOT / 'tfod_work' / 'training'
cmd = [sys.executable, str(ROOT / 'tf_models' / 'research' / 'object_detection' / 'exporter_main_v2.py'), '--input_type=image_tensor', f'--pipeline_config_path={pipeline_config.as_posix()}', f'--trained_checkpoint_dir={trained_checkpoint_dir.as_posix()}', f'--output_directory={export_dir.as_posix()}']
subprocess.run(cmd, env=env, cwd=str(ROOT), check=True)
saved_model_dir = export_dir / 'saved_model'
converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()
(EXPORT_DIR / 'ssd_mobilenetv2_float16.tflite').write_bytes(tflite_model)
print('TFLite float16 tersimpan')


CalledProcessError: Command '['c:\\Users\\TROYY\\anaconda3\\envs\\skripkir\\python.exe', 'd:\\Kuliah\\Skripsi\\model-dataset-2-kelas\\tf_models\\research\\object_detection\\exporter_main_v2.py', '--input_type=image_tensor', '--pipeline_config_path=d:/Kuliah/Skripsi/model-dataset-2-kelas/tfod_work/annotations/ssd_mobilenet_v2_fpnlite_lele.config', '--trained_checkpoint_dir=d:/Kuliah/Skripsi/model-dataset-2-kelas/tfod_work/training', '--output_directory=d:/Kuliah/Skripsi/model-dataset-2-kelas/tfod_work/export/ssd_savedmodel']' returned non-zero exit status 1.

## Catatan
- Dataset asli tetap di `dataset_split`.
- Copy kerja ada di `dataset_split_tfod`.
- Folder model lokal harus ada di `tfod_work/models/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8`.
